# ---------- Network building notebook ----------

The goal of this notebook is to build a weighted, directed graph of the London bike-sharing network.

Author: Artur Werys

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

import networkx as nx

In [ ]:
def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)
    for path in [current, *current.parents]:
        if (path / "Data").exists():
            return path
    raise FileNotFoundError("Could not find project root containing Data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "Data"

DATA_FILE = DATA_DIR / "final_trip_data.parquet"

In [ ]:
trip_data = pd.read_parquet(DATA_FILE)
trip_data.head()

## ---------- Stations as network nodes ----------

In [ ]:
start_stations_df = trip_data[
    [
        "start_station_id",
        "start_station_name",
        "start_lat",
        "start_lon",
    ]
].drop_duplicates()

start_stations_df = start_stations_df.rename(columns={
    "start_station_id": "station_id",
    "start_station_name": "station_name",
    "start_lat": "lat",
    "start_lon": "lon"
})

start_stations_df.head()

In [ ]:
end_stations_df = trip_data[
    [
        "end_station_id",
        "end_station_name",
        "end_lat",
        "end_lon",
    ]
].drop_duplicates()

end_stations_df = end_stations_df.rename(columns={
    "end_station_id": "station_id",
    "end_station_name": "station_name",
    "end_lat": "lat",
    "end_lon": "lon"
})

end_stations_df.head()

In [ ]:
stations_df = pd.concat([start_stations_df, end_stations_df], ignore_index=True).drop_duplicates()
print("Number of stations:", len(stations_df))

## ---------- Weighted directed edges ----------

In [ ]:
edges_df = trip_data.groupby(
    [
        "start_station_id",
        "start_station_name",
        "end_station_id",
        "end_station_name"
    ]
).agg(
    weight=("start_station_id", "count")
).reset_index()

In [ ]:
edges_df.head(-10)

## ---------- Building directed weighted graph ----------

In [ ]:
graph = nx.from_pandas_edgelist(
    edges_df,
    source="start_station_name",
    target="end_station_name",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

## ---------- Creating station positions dictionary ----------


In [ ]:
positions = {}

stations_df["lat"] = stations_df["lat"].astype(float)
stations_df["lon"] = stations_df["lon"].astype(float)

for _, row in stations_df.iterrows():

    positions[row["station_name"]] = (
        row["lon"],
        row["lat"]
    )